# EXHEART - leakage-free recalibration (standalone)

Run this once in Colab (GPU runtime). It mounts your Drive, retrains each stack from the
committed CSVs, and compares the **published (leaky)** Platt calibration against the
**leakage-free out-of-fold** calibration for the three self-contained arms:
BRFSS 2015 (development), BRFSS 2020 (retrained), and the clinical dataset.

For each arm it prints a before/after table and saves the leakage-free calibrator, metrics,
and calibrated test predictions (files suffixed `_leakagefree`). The base models, test split,
SHAP, fairness, and transport analyses are unaffected; only calibration changes.

The 2020 **transport** column reuses the 2015 calibrator this notebook recomputes, so it
changes negligibly; update it in EXHEART_02 by loading `models/brfss2015/platt_leakagefree.pkl`
instead of rebuilding the 2015 Platt scaler.

Expected: the numbers barely move (BRFSS 2015 ECE-post ~0.011; clinical ~0.027-0.028; AUC,
AUPRC, Brier, sensitivity, specificity unchanged within ~0.004).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, joblib, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

REPO = '/content/drive/MyDrive/EXHEART_Research/exheart-research'
SEED = 42

def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for i in range(n_bins):
        m = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if m.sum(): ece += m.sum() * abs(y_true[m].mean() - y_prob[m].mean())
    return ece / len(y_true)

def build_mlp(d):
    inp = keras.Input(shape=(d,))
    x = layers.Dense(256, activation='relu')(inp); x = layers.BatchNormalization()(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x);   x = layers.BatchNormalization()(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(64,  activation='relu')(x);   x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=[keras.metrics.AUC(name='auc')])
    return m

def _es():
    return keras.callbacks.EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True, mode='max')

def recalibrate_arm(X, y, pt, arm, models_dir, metrics_path, preds_path):
    """Train stack, compare leaky vs out-of-fold calibration, save leakage-free outputs."""
    print(f'\n=== {arm} : training stack (this is the slow part) ===')
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    Xtr_a, Xte_a, ytr_a, yte_a = Xtr.values, Xte.values, ytr.values, yte.values
    sc = StandardScaler(); Xtr_sc = sc.fit_transform(Xtr_a); Xte_sc = sc.transform(Xte_a)
    cw = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
    spw = cw[1] / cw[0]; kcw = {0: cw[0], 1: cw[1]}
    def trees():
        return (XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, scale_pos_weight=spw,
                              eval_metric='logloss', random_state=SEED, n_jobs=-1),
                LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, class_weight='balanced',
                               random_state=SEED, n_jobs=-1, verbose=-1),
                RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced',
                                       random_state=SEED, n_jobs=-1))
    # full base learners (used for test predictions - AUC/AUPRC identical to paper)
    xg, lg, rf = trees(); xg.fit(Xtr_a, ytr_a); lg.fit(Xtr_a, ytr_a); rf.fit(Xtr_a, ytr_a)
    tf.random.set_seed(SEED); mf = build_mlp(Xtr_sc.shape[1])
    mf.fit(Xtr_sc, ytr_a, epochs=50, batch_size=512, validation_split=0.1, class_weight=kcw, callbacks=[_es()], verbose=0)
    # 5-fold out-of-fold predictions (honest calibration data)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED); oof = np.zeros((len(Xtr), 4))
    for k, (tr, va) in enumerate(skf.split(Xtr_a, ytr_a)):
        a, b, c = trees(); a.fit(Xtr_a[tr], ytr_a[tr]); b.fit(Xtr_a[tr], ytr_a[tr]); c.fit(Xtr_a[tr], ytr_a[tr])
        tf.random.set_seed(SEED + k); mm = build_mlp(Xtr_sc.shape[1])
        mm.fit(Xtr_sc[tr], ytr_a[tr], epochs=50, batch_size=512, validation_split=0.1, class_weight=kcw, callbacks=[_es()], verbose=0)
        oof[va] = np.column_stack([a.predict_proba(Xtr_a[va])[:, 1], b.predict_proba(Xtr_a[va])[:, 1],
                                   c.predict_proba(Xtr_a[va])[:, 1], mm.predict(Xtr_sc[va], verbose=0).ravel()])
        print(f'  OOF fold {k+1}/5 done')
    meta = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED).fit(oof, ytr_a)
    raw_test = meta.predict_proba(np.column_stack([xg.predict_proba(Xte_a)[:, 1], lg.predict_proba(Xte_a)[:, 1],
                                                   rf.predict_proba(Xte_a)[:, 1], mf.predict(Xte_sc, verbose=0).ravel()]))[:, 1]
    # LEAKY calibrator (reproduces the published pipeline: in-sample slice of the training set)
    Xcal, _, ycal, _ = train_test_split(Xtr, ytr_a, test_size=0.8, random_state=SEED, stratify=ytr_a)
    cal_leaky = meta.predict_proba(np.column_stack([xg.predict_proba(Xcal.values)[:, 1], lg.predict_proba(Xcal.values)[:, 1],
                                                    rf.predict_proba(Xcal.values)[:, 1], mf.predict(sc.transform(Xcal.values), verbose=0).ravel()]))[:, 1]
    ycal_a = ycal.values if hasattr(ycal, 'values') else ycal
    platt_leaky = LogisticRegression(max_iter=1000).fit(cal_leaky.reshape(-1, 1), ycal_a)
    # HONEST calibrator (out-of-fold stack predictions)
    platt_honest = LogisticRegression(max_iter=1000).fit(meta.predict_proba(oof)[:, 1].reshape(-1, 1), ytr_a)
    def row(prob):
        pred = (prob >= pt).astype(int); tn, fp, fn, tp = confusion_matrix(yte_a, pred).ravel()
        return dict(AUC=round(roc_auc_score(yte_a, prob), 3), AUPRC=round(average_precision_score(yte_a, prob), 3),
                    Brier=round(brier_score_loss(yte_a, prob), 3), ECE=round(compute_ece(yte_a, prob), 3),
                    Sens=round(tp/(tp+fn), 3), Spec=round(tn/(tn+fp), 3))
    leaky = platt_leaky.predict_proba(raw_test.reshape(-1, 1))[:, 1]
    honest = platt_honest.predict_proba(raw_test.reshape(-1, 1))[:, 1]
    ece_pre = round(compute_ece(yte_a, raw_test), 3)
    mL, mH = row(leaky), row(honest)
    print(f'  {arm}: ECE-pre={ece_pre} | ECE-post leaky={mL["ECE"]} -> honest={mH["ECE"]}')
    # save leakage-free outputs
    os.makedirs(models_dir, exist_ok=True); os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
    joblib.dump(platt_honest, os.path.join(models_dir, 'platt_leakagefree.pkl'))
    out = dict(arm=arm, threshold=pt, ECE_pre_calibration=ece_pre,
               published_leaky=mL, leakage_free=mH, calibration='out-of-fold cross-fitting')
    json.dump(out, open(metrics_path, 'w'), indent=2)
    pd.DataFrame({'y_true': yte_a, 'y_prob_leakagefree': honest}).to_csv(preds_path, index=False)
    return dict(arm=arm, ece_pre=ece_pre, leaky=mL, honest=mH)

print('Setup done. GPU:', tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
Setup done. GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ---------- BRFSS 2015 (development, pt=0.12) ----------
df = pd.read_csv(f'{REPO}/data/brfss2015/heart_disease_health_indicators_BRFSS2015.csv')
X = df.drop(columns=['HeartDiseaseorAttack']); y = df['HeartDiseaseorAttack'].astype(int)
R15 = recalibrate_arm(
    X, y, pt=0.12, arm='BRFSS 2015 (development)',
    models_dir=f'{REPO}/models/brfss2015',
    metrics_path=f'{REPO}/results/brfss2015/tables/metrics_leakagefree.json',
    preds_path=f'{REPO}/results/brfss2015/brfss2015_test_predictions_leakagefree.csv')


=== BRFSS 2015 (development) : training stack (this is the slow part) ===
  OOF fold 1/5 done
  OOF fold 2/5 done
  OOF fold 3/5 done
  OOF fold 4/5 done
  OOF fold 5/5 done
  BRFSS 2015 (development): ECE-pre=0.259 | ECE-post leaky=0.011 -> honest=0.011


In [ ]:
# ---------- BRFSS 2020 (retrained / independent, pt=0.12) ----------
df20 = pd.read_csv(f'{REPO}/data/brfss2020/heart_2020_cleaned.csv')
y20 = (df20['HeartDisease'] == 'Yes').astype(int)
X20 = df20.drop(columns=['HeartDisease']).copy()
for col in X20.select_dtypes(include='object').columns:      # label-encode categoricals (as in NB02)
    X20[col] = pd.factorize(X20[col])[0]
R20 = recalibrate_arm(
    X20, y20, pt=0.12, arm='BRFSS 2020 (retrained)',
    models_dir=f'{REPO}/models/brfss2020',
    metrics_path=f'{REPO}/results/brfss2020/independent_pipeline/tables/metrics_leakagefree.json',
    preds_path=f'{REPO}/results/brfss2020/independent_pipeline/brfss2020_test_predictions_leakagefree.csv')

# NOTE: if your NB02 uses a specific label-encoding order for 2020 categoricals, the AUC here
# may differ by a hair from Table 3. What matters for this check is ECE-post leaky vs honest.


=== BRFSS 2020 (retrained) : training stack (this is the slow part) ===
  OOF fold 1/5 done
  OOF fold 2/5 done
  OOF fold 3/5 done
  OOF fold 4/5 done
  OOF fold 5/5 done
  BRFSS 2020 (retrained): ECE-pre=0.267 | ECE-post leaky=0.012 -> honest=0.011


In [ ]:
# ---------- Clinical dataset (pt=0.50) ----------
dfc = pd.read_csv(f'{REPO}/data/cardio/cardio_train.csv', sep=';').drop(columns=['id'])
dfc['age'] = (dfc['age'] / 365.25).round(1)
dfc['bmi'] = (dfc['weight'] / (dfc['height']/100)**2).round(1)
dfc = dfc[(dfc.ap_hi>=60)&(dfc.ap_hi<=250)&(dfc.ap_lo>=40)&(dfc.ap_lo<=160)&
          (dfc.height>=100)&(dfc.height<=220)&(dfc.weight>=30)&(dfc.weight<=200)].reset_index(drop=True)
MC = ['age','gender','height','weight','ap_hi','ap_lo','cholesterol','gluc','smoke','alco','active','bmi']
Xc = dfc[MC]; yc = dfc['cardio']
RC = recalibrate_arm(
    Xc, yc, pt=0.50, arm='Clinical (cardio)',
    models_dir=f'{REPO}/models/cardio',
    metrics_path=f'{REPO}/results/cardio/tables/metrics_leakagefree.json',
    preds_path=f'{REPO}/results/cardio/cardio_test_predictions_leakagefree.csv')

In [ ]:
# ---------- BEFORE / AFTER SUMMARY ----------
print('\n' + '='*70)
print(f'{"arm":<26}{"metric":<10}{"PUBLISHED":>12}{"LEAK-FREE":>12}')
print('='*70)
for R in [R15, R20, RC]:
    for key in ['ECE','AUC','AUPRC','Brier','Sens','Spec']:
        pub = R['leaky'][key]; new = R['honest'][key]
        mark = '' if abs(pub-new) < 0.005 else '  <-- check'
        label = R['arm'] if key=='ECE' else ''
        print(f'{label:<26}{key:<10}{pub:>12}{new:>12}{mark}')
    print('-'*70)
print('\nSaved per arm: platt_leakagefree.pkl, metrics_leakagefree.json, *_predictions_leakagefree.csv')
print('Commit these, then paste the three metrics_leakagefree.json contents back.')